# Baseline

Наша задача - посмотреть, что именно вносит мультимодальность и какие методы наиболее пригодны для улучшения метрик.

Для этого нам необходимо понимать бейзлайны по разным модальностям.

In [1]:
%%capture
!pip install catboost transformers torch torchvision timm

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Шаг 1: Загрузка данных и первичный осмотр

In [3]:
from pathlib import Path

DATA_DIR = Path("/Users/sofya/Desktop/ВКР /data")
assert DATA_DIR.exists(), f"DATA_DIR не найден: {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)

DATA_DIR: /Users/sofya/Desktop/ВКР /data


In [4]:
df = pd.read_csv(DATA_DIR / "ml_ozon_ounterfeit_train.csv", encoding="utf-8")
print("df shape:", df.shape)


df shape: (197198, 45)


**Наши данные**: 197,198 наблюдений, 52 признака

**Типы данных**:

- 30 числовых признаков (float64)

- 17 целочисленных признаков (int64)

- 4 текстовых признака (object)

- 1 категориальный (category)

**Ключевые группы признаков:**

- Идентификаторы: id, ItemID, SellerID

- Целевая переменная: resolution

- Временные признаки: item_time_alive, seller_time_alive

- Рейтинги и отзывы: rating_X_count, comments_published_count

- Продажи и возвраты: item_count_salesX, item_count_returnsX

- Финансовые метрики: GmvTotalX, ExemplarReturnedValueTotalX

- Текстовые данные: brand_name, description, name_rus

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
trainval_idx, test_idx = next(gss1.split(df, df["resolution"], groups=df["SellerID"]))
trainval_df = df.iloc[trainval_idx].copy()
test_df = df.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
tr_idx, val_idx = next(
    gss2.split(trainval_df, trainval_df["resolution"], groups=trainval_df["SellerID"])
)
train_df = trainval_df.iloc[tr_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

assert set(train_df["SellerID"]).isdisjoint(set(val_df["SellerID"]))
assert set(train_df["SellerID"]).isdisjoint(set(test_df["SellerID"]))
assert set(val_df["SellerID"]).isdisjoint(set(test_df["SellerID"]))

print("Train:", train_df.shape, "rate:", round(train_df["resolution"].mean(), 4))
print("Val:  ", val_df.shape,   "rate:", round(val_df["resolution"].mean(), 4))
print("Test: ", test_df.shape,  "rate:", round(test_df["resolution"].mean(), 4))

Train shape: (177380, 45)
Test shape: (19818, 45)
Train counterfeit rate: 0.05882850377720149
Test counterfeit rate: 0.13205167019880917


In [ ]:
train_df = train_df.copy()
test_df = test_df.copy()

В качестве метрик возьмем ROC-AUC и PR-AUC.

# baseline A - только текст

сначала посмотрим, какие результаты нам даст анализ исключительно текста. на озон капе это принесло неплохие результаты

In [ ]:
train_df["text"] = (
    train_df["name_rus"].fillna("") + " " +
    train_df["description"].fillna("") + " " +
    train_df["brand_name"].fillna("")
)

test_df["text"] = (
    test_df["name_rus"].fillna("") + " " +
    test_df["description"].fillna("") + " " +
    test_df["brand_name"].fillna("")
)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=5
)

X_train_text = vectorizer.fit_transform(train_df["text"])
X_test_text = vectorizer.transform(test_df["text"])

y_train = train_df["resolution"]
y_test = test_df["resolution"]

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_text, y_train)

y_pred_proba = model.predict_proba(X_test_text)[:,1]

roc = roc_auc_score(y_test, y_pred_proba)
pr = average_precision_score(y_test, y_pred_proba)

print("ROC-AUC:", roc)
print("PR-AUC:", pr)

ROC-AUC: 0.9050942599888389
PR-AUC: 0.5903242214242288


ROC-AUC: 0.9050942599888389
PR-AUC: 0.5903242214242288


В качестве первого baseline была обучена линейная модель на текстовых признаках.
При seller-based разбиении данных достигнут ROC-AUC 0.905, PR-AUC 0.59.
Это свидетельствует о высокой информативности текстового контента карточек товаров.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = model.coef_[0]

top_positive_idx = np.argsort(coef)[-20:]
top_negative_idx = np.argsort(coef)[:20]

print("топ слов в подделках:")
print(feature_names[top_positive_idx])

print("топ слов в оригиналах:")
print(feature_names[top_negative_idx])

Top counterfeit words:
['экшн камера' 'оперативная память' 'аккумулятор xiaomi' 'черный'
 'устройств xiaomi' 'белый' 'оригинальный' 'роутер' 'пылесос' 'pioneer'
 'картридж hp' 'геймпад' 'крышка для' 'колонка' 'монитор' 'кт' 'наушники'
 'картридж samsung' 'батарейка' 'mivis']

Top original words:
['для xiaomi' 'игра' 'для' 'для apple' 'gps' 'для samsung' 'galaxy'
 'realme' 'kitfort' 'samsung' 'для huawei' 'infinix' 'electrolux' 'baseus'
 'пылесоса' 'шлейф' 'xiaomi' 'для ноутбука' 'ballu' 'galaxy line']


# Baseline 2 - CatBoost
теперь оценим табличные данные кэтбустом.

In [ ]:
train_df = train_df.copy()
test_df = test_df.copy()

target = "resolution"

drop_cols = [
    "id",
    "ItemID",
    "SellerID",
    "name_rus",
    "description",
    "brand_name",
    "text"
]

feature_cols = [col for col in train_df.columns if col not in drop_cols + [target]]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df[target]
y_test = test_df[target]

print("Фичей:", len(feature_cols))

Num features: 38


In [ ]:
#Заполняем только числовые пропуски нулями
num_cols = X_train.select_dtypes(include=["float64", "int64"]).columns

X_train[num_cols] = X_train[num_cols].fillna(0)
X_test[num_cols] = X_test[num_cols].fillna(0)

/tmp/ipython-input-567/1902166372.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[num_cols] = X_train[num_cols].fillna(0)
/tmp/ipython-input-567/1902166372.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[num_cols] = X_test[num_cols].fillna(0)


In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", len(cat_cols))
print("Numerical:", len(num_cols))

Categorical: 1
Numerical: 37


In [ ]:
X_val = val_df[feature_cols]
y_val = val_df[target]
X_val[num_cols] = X_val[num_cols].fillna(0)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_cb = CatBoostClassifier(
    iterations=700,
    depth=6,
    learning_rate=0.05,
    eval_metric="AUC",
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=100
)

model_cb.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    use_best_model=True
)

y_pred_proba_cb = model_cb.predict_proba(X_test)[:, 1]

roc_cb = roc_auc_score(y_test, y_pred_proba_cb)
pr_cb = average_precision_score(y_test, y_pred_proba_cb)

print("CatBoost ROC-AUC:", roc_cb)
print("CatBoost PR-AUC:", pr_cb)

0:	test: 0.8765860	best: 0.8765860 (0)	total: 472ms	remaining: 5m 29s
100:	test: 0.9009133	best: 0.9050578 (83)	total: 36.2s	remaining: 3m 34s
200:	test: 0.8990093	best: 0.9050578 (83)	total: 56.1s	remaining: 2m 19s
300:	test: 0.8985102	best: 0.9050578 (83)	total: 1m 16s	remaining: 1m 41s
400:	test: 0.9009826	best: 0.9050578 (83)	total: 1m 35s	remaining: 1m 11s
500:	test: 0.9000154	best: 0.9050578 (83)	total: 1m 58s	remaining: 47s
600:	test: 0.8996653	best: 0.9050578 (83)	total: 2m 17s	remaining: 22.7s
699:	test: 0.8983994	best: 0.9050578 (83)	total: 2m 36s	remaining: 0us

bestTest = 0.9050578499
bestIteration = 83

Shrink model to first 84 iterations.
CatBoost ROC-AUC: 0.9050578499170622
CatBoost PR-AUC: 0.6230967508174646


CatBoost ROC-AUC: 0.9050578499170622
CatBoost PR-AUC: 0.6230967508174646

Текст по метрикам схож с табличными данными.

# Baseline 3 - мультимодальный с эмбеддингами

In [ ]:
EMB_PATH = Path("/Users/sofya/Desktop/ВКР /diplomahse/image_embeddings.csv")
assert EMB_PATH.exists(), f"EMB_PATH не найден: {EMB_PATH}"
print("EMB_PATH:", EMB_PATH)

In [ ]:
# Загрузка эмбеддингов
image_emb = pd.read_csv(EMB_PATH)
print("Эмбеддинги загружены:", image_emb.shape)

Эмбеддинги загружены: (219195, 3)


In [ ]:
#Убедимся, что ключ называется одинаково (привести к нижнему регистру)
image_emb.columns = [c.lower() for c in image_emb.columns]

In [ ]:
print("Строк в image_emb:", len(image_emb))
print("Строк в df:", len(df))

Строк в image_emb: 219195
Строк в df (весь трейн): 197198


In [ ]:
image_emb.head()

,unnamed: 0,image_name,embedding
0,0,10.png,[-0.3904604 -0.22580202 -0.14542326 -0.032227...
1,1,100.png,[ 0.28374434 -0.14928259 0.29042315 -0.312558...
2,2,10000.png,[ 0.08850661 -0.42001283 0.4141257 -0.210395...
3,3,100009.png,[ 0.22624922 -0.3577845 0.43859392 -0.295216...
4,4,100012.png,[-0.19313276 0.09382293 -0.25836873 0.104856...


In [ ]:
#Извлекаем item_id из имени файла
image_emb['item_id'] = image_emb['image_name'].str.replace('.png', '').astype(int)

print(image_emb['item_id'].head())
print(df['ItemID'].head())  # посмотри как выглядит ItemID в df — тот же формат?

0        10
1       100
2     10000
3    100009
4    100012
Name: item_id, dtype: int64
0     78312
1    141999
2     53306
3    202599
4    163725
Name: ItemID, dtype: int64


In [ ]:
image_emb_train = image_emb[image_emb['item_id'].isin(df['ItemID'])]
print("Было в image_emb:", len(image_emb))
print("Осталось после фильтрации:", len(image_emb_train))

Было в image_emb: 219195
Осталось после фильтрации: 196460


In [ ]:
# Используем уже готовое seller-based разбиение
train_image_emb = image_emb_train[image_emb_train['item_id'].isin(train_df['ItemID'])]
test_image_emb  = image_emb_train[image_emb_train['item_id'].isin(test_df['ItemID'])]

print("Train эмбеддинги:", len(train_image_emb))
print("Test эмбеддинги:", len(test_image_emb))

Train эмбеддинги: 176646
Test эмбеддинги: 19814


In [ ]:
def parse_embedding(s):
    # убираем скобки и парсим через пробел
    s = s.strip().strip('[]')
    return np.fromstring(s, sep=' ')

X_train_img = np.vstack(train_image_emb['embedding'].apply(parse_embedding))
X_test_img  = np.vstack(test_image_emb['embedding'].apply(parse_embedding))

print("X_train_img shape:", X_train_img.shape)
print("X_test_img shape:", X_test_img.shape)

X_train_img shape: (176646, 32)
X_test_img shape: (19814, 32)


In [ ]:
# y берём через merge чтобы порядок совпадал с X
y_train_img = train_image_emb.merge(
    train_df[['ItemID', 'resolution']],
    left_on='item_id', right_on='ItemID'
)['resolution'].values

y_test_img = test_image_emb.merge(
    test_df[['ItemID', 'resolution']],
    left_on='item_id', right_on='ItemID'
)['resolution'].values

print("y_train counterfeit rate:", y_train_img.mean().round(4))
print("y_test counterfeit rate:", y_test_img.mean().round(4))
print("Размеры совпадают:", len(X_train_img) == len(y_train_img), len(X_test_img) == len(y_test_img))

y_train counterfeit rate: 0.0586
y_test counterfeit rate: 0.1319
Размеры совпадают: True True


In [ ]:
#масштабировуем
scaler = StandardScaler()
X_train_img_scaled = scaler.fit_transform(X_train_img)
X_test_img_scaled  = scaler.transform(X_test_img)

# делаем простую регрессию, но с учётом дисбаланса классов
model_img = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_img.fit(X_train_img_scaled, y_train_img)

# Метрики
y_pred_proba_img = model_img.predict_proba(X_test_img_scaled)[:, 1]

roc = roc_auc_score(y_test_img, y_pred_proba_img)
pr  = average_precision_score(y_test_img, y_pred_proba_img)

precision, recall, _ = precision_recall_curve(y_test_img, y_pred_proba_img)
mask = precision >= 0.9
recall_at_90 = recall[mask].max() if mask.any() else 0.0

print(f"Image-only ROC-AUC:        {roc:.4f}")
print(f"Image-only PR-AUC:         {pr:.4f}")
print(f"Recall @ Precision >= 0.9: {recall_at_90:.4f}")

Image-only ROC-AUC:        0.8532
Image-only PR-AUC:         0.4616
Recall @ Precision >= 0.9: 0.0042


## Выводы по baseline-моделям

Были обучены три baseline-модели на разных модальностях данных
с единым seller-based разбиением (train/test), чтобы избежать утечки данных.

**Результаты:**

| Baseline | ROC-AUC | PR-AUC | Recall@P≥0.9 |
|---|---|---|---|
| Текст (TF-IDF + LR) | 0.9051 | 0.5903 | ~0.0 |
| CatBoost (табличные признаки) | 0.9051 | 0.6231 | 0.0145 |
| Изображения (эмбеддинги + LR) | 0.8532 | 0.4616 | 0.0042 |

**Наблюдения:**

1. Текст и табличные признаки дают сопоставимый ROC-AUC (~0.90),
что говорит о высокой информативности обеих модальностей по отдельности.

2. CatBoost превосходит текстовую модель по PR-AUC (0.62 vs 0.59) —
табличные признаки лучше справляются с дисбалансом классов.

3. Изображения в текущем виде (32-мерные эмбеддинги) уступают остальным
модальностям.

4. Критическая метрика Recall@Precision≥0.9 близка к нулю у всех трёх моделей.
Это означает, что ни одна одномодальная модель не способна уверенно
детектировать контрафакт без большого числа ложных срабатываний —
что неприемлемо для production-системы модерации.

**Вывод:** одномодальные baseline-модели подтверждают информативность
каждого источника данных, но не решают задачу в полной мере.
Это мотивирует переход к мультимодальному подходу,
где фьюжн текста, табличных признаков и изображений
потенциально способен улучшить Recall@P≥0.9. Интересно будет посмотреть, какие методы принесут наилучший результат относительно этих цифр.